# Haystack: Scalable Search & Answer Pipelines
### Components, Retrievers, Pipelines, and Evaluation

**Project -- NLP / Information Retrieval Portfolio**

This notebook implements a complete **Haystack-style** search-and-answer system from scratch, demonstrating:

1. **Components** -- the atomic units of processing (preprocessors, embedders, retrievers, readers)
2. **Document Store** -- the persistence layer connecting ingestion to retrieval
3. **Retrievers** -- BM25 (sparse), dense embedding, and hybrid strategies compared head-to-head
4. **Pipelines** -- declarative DAGs that wire components together for indexing and querying
5. **Evaluation** -- measuring retrieval quality and answer accuracy with standard IR metrics

All components run locally with simulated data and embeddings. No API keys required.

## What is Haystack?

[Haystack](https://haystack.deepset.ai/) (by deepset) is a framework for building production-grade NLP pipelines. Its key design principles:

| Principle | How Haystack Implements It |
|---|---|
| **Composability** | Every operation is a `Component` with typed inputs/outputs |
| **Swappability** | Swap retrievers, LLMs, or stores without changing pipeline code |
| **Scalability** | Document stores scale from in-memory to Elasticsearch/Weaviate/Pinecone |
| **Evaluation-first** | Built-in metrics for retrieval and generation quality |
| **Production-ready** | REST API, logging, and serialisation out of the box |

### Haystack 2.x Architecture

```
                        Pipeline (DAG)
     .--------------------------------------------------.
     |                                                    |
     |   [Preprocessor] --> [Embedder] --> [DocStore]    |   Indexing
     |                                                    |
     '----------------------------------------------------'

     .--------------------------------------------------.
     |                                                    |
     |   [Query] --> [Retriever] --> [Reader/Generator]  |   Querying
     |                                                    |
     '----------------------------------------------------'
```

### Component Protocol

Every Haystack component implements:
- `@component` decorator (registers it in the pipeline graph)
- `run(**kwargs) -> dict` method (processes inputs, returns outputs)
- Input/output type declarations (enables pipeline validation before execution)

## Learning Objectives

1. Understand Haystack's **component protocol** and how components compose into pipelines
2. Build a **Document Store** with indexing, filtering, and retrieval capabilities
3. Implement **three retriever strategies**: BM25 (sparse), dense (embedding), and hybrid
4. Construct **indexing and querying pipelines** as directed acyclic graphs
5. Implement a **Reader** component that extracts answers from retrieved documents
6. Design a **retrieval evaluation framework** with MRR, Recall@K, nDCG, and F1 metrics
7. Compare retriever performance across query types and difficulty levels

## Problem Statement

Build a question-answering system over a **technical knowledge base** (cloud infrastructure documentation) that:

1. **Indexes** 25+ documents through a preprocessing pipeline (cleaning, splitting, embedding)
2. **Retrieves** relevant passages for natural language questions using three strategies
3. **Extracts answers** from retrieved passages with confidence scores
4. **Evaluates** each retriever with 30 labelled queries and ground-truth relevance judgements
5. **Recommends** which retriever to use for which query type based on measured performance

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ["pandas", "numpy", "matplotlib", "seaborn", "plotly"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Environment ready.")

## Imports & Configuration

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import hashlib, json, time, uuid, math, re, textwrap, copy
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict
from typing import Any, Optional, Protocol
from enum import Enum
from collections import defaultdict, Counter
from abc import ABC, abstractmethod

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
np.random.seed(42)
print("Imports OK")

## Document Model

Haystack's `Document` is the universal data container. Every component reads and writes Documents.

In [ ]:
@dataclass
class Document:
    """Core data unit -- mirrors haystack.dataclasses.Document.

    In production Haystack:
        from haystack.dataclasses import Document
        doc = Document(content="...", meta={"source": "wiki"})
    """
    id: str = ""
    content: str = ""
    meta: dict = field(default_factory=dict)
    embedding: Optional[np.ndarray] = None
    score: Optional[float] = None

    def __post_init__(self):
        if not self.id:
            self.id = hashlib.sha256(self.content.encode()).hexdigest()[:12]

    @property
    def content_length(self) -> int:
        return len(self.content)

    def to_dict(self) -> dict:
        d = {"id": self.id, "content": self.content, "meta": self.meta,
             "score": self.score}
        if self.embedding is not None:
            d["embedding_dim"] = len(self.embedding)
        return d

    def __repr__(self):
        return (f"Document(id={self.id}, len={self.content_length}, "
                f"meta_keys={list(self.meta.keys())})")


# Quick test
doc = Document(content="Kubernetes pods are the smallest deployable units.",
               meta={"source": "k8s-docs", "section": "concepts"})
print(doc)
print(f"  Content hash ID: {doc.id}")

## Component Protocol

Every Haystack component follows the same contract. This enables:
- **Pipeline validation**: the framework checks that outputs of component A match inputs of component B before running
- **Serialisation**: pipelines can be saved/loaded as YAML
- **Swappability**: replace any component without changing the pipeline structure

```python
# Haystack 2.x component protocol:
@component
class MyComponent:
    @component.output_types(documents=list[Document])
    def run(self, texts: list[str]) -> dict:
        docs = [Document(content=t) for t in texts]
        return {"documents": docs}
```

In [ ]:
class Component(ABC):
    """Base class for all pipeline components.

    Mirrors the Haystack @component protocol:
    - Each component declares input/output types
    - run() accepts kwargs, returns a dict of named outputs
    - Components are stateless between run() calls (state lives in the pipeline)
    """

    @property
    def component_name(self) -> str:
        return self.__class__.__name__

    @abstractmethod
    def run(self, **kwargs) -> dict:
        """Process inputs and return named outputs."""
        ...

    def __repr__(self):
        return f"{self.component_name}()"


print("Component base class defined.")

## Knowledge Base: Cloud Infrastructure Documentation

We build a corpus of 25 technical documents covering Kubernetes, AWS, networking, security, databases, and CI/CD -- the kind of knowledge base engineers query daily.

In [ ]:
CORPUS = [
    # -- Kubernetes --
    {"title": "Kubernetes Pod Lifecycle",
     "content": (
         "A Pod is the smallest deployable unit in Kubernetes. Pods go through phases: "
         "Pending (waiting for scheduling), Running (at least one container active), "
         "Succeeded (all containers terminated successfully), Failed (at least one "
         "container failed), and Unknown (node communication lost). The kubelet on each "
         "node monitors pod health via liveness probes (HTTP, TCP, or exec checks) and "
         "readiness probes that control traffic routing. Restart policies: Always "
         "(default), OnFailure, or Never. Init containers run before app containers "
         "and must complete successfully. Pods are ephemeral -- use Deployments for "
         "durability."
     ), "category": "kubernetes", "difficulty": "beginner"},

    {"title": "Kubernetes Services and Networking",
     "content": (
         "Services provide stable network endpoints for Pods. ClusterIP (default) "
         "exposes the service on an internal IP. NodePort exposes on each node's IP "
         "at a static port (30000-32767). LoadBalancer provisions a cloud load balancer. "
         "ExternalName maps to a DNS name. Ingress controllers (nginx, Traefik, AWS ALB) "
         "manage HTTP/HTTPS routing with path-based and host-based rules. Network "
         "Policies restrict pod-to-pod traffic using label selectors. CoreDNS handles "
         "cluster DNS resolution: service.namespace.svc.cluster.local."
     ), "category": "kubernetes", "difficulty": "intermediate"},

    {"title": "Kubernetes Deployments and Scaling",
     "content": (
         "Deployments manage ReplicaSets which manage Pods. Rolling updates replace pods "
         "incrementally: maxUnavailable and maxSurge control the pace. Rollback with "
         "'kubectl rollout undo'. Horizontal Pod Autoscaler (HPA) scales replicas based "
         "on CPU/memory or custom metrics. Vertical Pod Autoscaler (VPA) adjusts resource "
         "requests/limits. Cluster Autoscaler adds/removes nodes based on pending pods. "
         "DaemonSets ensure one pod per node (logging, monitoring agents). StatefulSets "
         "provide stable network IDs and persistent storage for databases."
     ), "category": "kubernetes", "difficulty": "intermediate"},

    {"title": "Kubernetes ConfigMaps and Secrets",
     "content": (
         "ConfigMaps store non-sensitive configuration as key-value pairs. Mount as "
         "environment variables or volumes. Secrets store sensitive data (passwords, "
         "tokens, certificates) base64-encoded. Types: Opaque (default), "
         "kubernetes.io/tls, kubernetes.io/dockerconfigjson. Best practices: use "
         "external secret managers (Vault, AWS Secrets Manager) via CSI driver or "
         "External Secrets Operator. Never commit secrets to Git. Rotate secrets "
         "regularly. Use RBAC to restrict secret access to specific namespaces."
     ), "category": "kubernetes", "difficulty": "intermediate"},

    {"title": "Helm Charts and Package Management",
     "content": (
         "Helm is the package manager for Kubernetes. Charts are bundles of YAML "
         "templates with configurable values.yaml. Commands: helm install, upgrade, "
         "rollback, uninstall. Chart repositories (Artifact Hub) host reusable charts. "
         "Helm 3 removed Tiller (security improvement). Chart dependencies managed "
         "via Chart.yaml. Values can be overridden with --set or -f custom-values.yaml. "
         "Hooks (pre-install, post-upgrade) run jobs at lifecycle points. "
         "OCI registry support allows storing charts in container registries."
     ), "category": "kubernetes", "difficulty": "advanced"},

    # -- AWS --
    {"title": "AWS EC2 Instance Types and Pricing",
     "content": (
         "EC2 offers instance families: General Purpose (M, T series), Compute Optimized "
         "(C series), Memory Optimized (R, X series), Storage Optimized (I, D series), "
         "and Accelerated Computing (P, G series for GPU). T instances use burstable "
         "credits. Pricing models: On-Demand (pay per second), Reserved (1-3 year commit, "
         "up to 72% savings), Spot (up to 90% savings, can be interrupted), Savings Plans "
         "(flexible commitment). Right-sizing: use AWS Compute Optimizer. Key metrics: "
         "vCPUs, memory GiB, network bandwidth Gbps, storage IOPS."
     ), "category": "aws", "difficulty": "beginner"},

    {"title": "AWS IAM Policies and Security",
     "content": (
         "IAM controls access to AWS resources. Policies are JSON documents with Effect "
         "(Allow/Deny), Action, Resource, and optional Condition. Policy types: "
         "identity-based (attached to users/roles/groups), resource-based (attached to "
         "S3 buckets, SQS queues), permissions boundaries, SCPs (organisation-wide). "
         "Best practices: least privilege principle, enable MFA, use roles instead of "
         "long-term access keys, rotate credentials every 90 days, use AWS CloudTrail "
         "for audit logging. IAM Access Analyzer identifies overly permissive policies."
     ), "category": "aws", "difficulty": "intermediate"},

    {"title": "AWS VPC and Network Architecture",
     "content": (
         "VPC provides isolated network environments. Subnets are public (internet "
         "gateway route) or private (NAT gateway for outbound). Security Groups are "
         "stateful firewalls at instance level. NACLs are stateless at subnet level. "
         "VPC Peering connects VPCs across accounts/regions. Transit Gateway centralises "
         "connectivity for multiple VPCs. PrivateLink provides private access to AWS "
         "services without traversing the internet. VPC Flow Logs capture IP traffic "
         "for analysis. CIDR planning: /16 VPC provides 65,536 IPs."
     ), "category": "aws", "difficulty": "intermediate"},

    {"title": "AWS Lambda and Serverless Computing",
     "content": (
         "Lambda runs functions without server management. Triggers: API Gateway, S3 "
         "events, SQS, DynamoDB Streams, EventBridge, CloudWatch Events. Limits: 15 min "
         "max timeout, 10 GB memory, 250 MB deployment package (unzipped). Cold starts "
         "add latency (100ms-10s depending on runtime). Provisioned concurrency eliminates "
         "cold starts for critical paths. Lambda@Edge runs at CloudFront edge locations. "
         "Pricing: per request ($0.20/million) + duration (GB-seconds). Use Powertools "
         "for structured logging, tracing, and metrics."
     ), "category": "aws", "difficulty": "intermediate"},

    {"title": "AWS RDS and Database Services",
     "content": (
         "RDS provides managed relational databases: MySQL, PostgreSQL, MariaDB, Oracle, "
         "SQL Server, and Aurora. Multi-AZ deployments provide automatic failover. Read "
         "replicas handle read-heavy workloads (up to 15 for Aurora). Aurora Serverless "
         "auto-scales compute capacity. Backup: automated daily snapshots with 35-day "
         "retention. Encryption at rest via KMS. Performance Insights monitors database "
         "load. DynamoDB for NoSQL: single-digit millisecond latency, auto-scaling, "
         "global tables for multi-region. ElastiCache for Redis/Memcached caching layer."
     ), "category": "aws", "difficulty": "intermediate"},

    # -- Networking --
    {"title": "DNS Resolution and Record Types",
     "content": (
         "DNS translates domain names to IP addresses. Record types: A (IPv4), AAAA "
         "(IPv6), CNAME (alias to another domain), MX (mail exchange), TXT (text data, "
         "SPF/DKIM), NS (name server), SOA (start of authority), SRV (service locator), "
         "CAA (certificate authority authorisation). TTL controls caching duration. "
         "Resolution path: recursive resolver to root servers to TLD servers to "
         "authoritative servers. Route 53 routing policies: simple, weighted, latency, "
         "failover, geolocation, multivalue answer."
     ), "category": "networking", "difficulty": "beginner"},

    {"title": "TLS/SSL Certificates and HTTPS",
     "content": (
         "TLS 1.3 secures data in transit. Handshake: ClientHello, ServerHello, "
         "certificate exchange, key agreement (ECDHE), finished. Certificate chain: "
         "server cert signed by intermediate CA signed by root CA. Certificate types: "
         "DV (domain validation), OV (organisation), EV (extended validation). "
         "ACME protocol (Let's Encrypt) automates certificate issuance. HSTS header "
         "forces HTTPS. OCSP stapling checks certificate revocation. Certificate "
         "pinning prevents MITM attacks. AWS ACM manages certificates for ALB/CloudFront."
     ), "category": "networking", "difficulty": "intermediate"},

    {"title": "Load Balancing Strategies",
     "content": (
         "Load balancers distribute traffic across backends. Algorithms: round-robin, "
         "least connections, IP hash, weighted round-robin. Layer 4 (TCP/UDP) vs Layer 7 "
         "(HTTP) balancing. AWS ALB: path-based routing, host-based routing, WebSocket "
         "support, WAF integration. NLB: ultra-low latency, static IP, TCP/UDP. Health "
         "checks: HTTP status code, TCP connection, interval/threshold configuration. "
         "Session affinity (sticky sessions) via cookies. Connection draining allows "
         "in-flight requests to complete during scaling events."
     ), "category": "networking", "difficulty": "intermediate"},

    # -- Security --
    {"title": "OWASP Top 10 Web Security Risks",
     "content": (
         "OWASP Top 10 (2021): A01 Broken Access Control (most common), A02 Cryptographic "
         "Failures, A03 Injection (SQL, XSS, command), A04 Insecure Design, A05 Security "
         "Misconfiguration, A06 Vulnerable Components, A07 Authentication Failures, "
         "A08 Software Integrity Failures, A09 Logging and Monitoring Failures, A10 SSRF. "
         "Mitigations: parameterised queries, input validation, CSP headers, CORS policies, "
         "rate limiting, WAF rules, dependency scanning (Snyk, Dependabot), SAST/DAST "
         "in CI pipeline. Security headers: X-Content-Type-Options, X-Frame-Options."
     ), "category": "security", "difficulty": "intermediate"},

    {"title": "Zero Trust Architecture",
     "content": (
         "Zero Trust assumes no implicit trust: verify every request regardless of network "
         "location. Principles: least privilege access, micro-segmentation, continuous "
         "verification, assume breach. Implementation: identity-aware proxy (BeyondCorp), "
         "mTLS between services, short-lived credentials (SPIFFE/SPIRE), device posture "
         "checks. Network micro-segmentation via service mesh (Istio, Linkerd). "
         "Context-aware access decisions based on user identity, device health, location, "
         "and risk score. Never trust, always verify."
     ), "category": "security", "difficulty": "advanced"},

    {"title": "Container Security Best Practices",
     "content": (
         "Container security spans build, deploy, and runtime. Build: use minimal base "
         "images (distroless, Alpine), scan for CVEs (Trivy, Snyk), avoid running as "
         "root, multi-stage builds to reduce attack surface. Deploy: use image signing "
         "(cosign, Notary), enforce policies (OPA Gatekeeper, Kyverno), enable Pod "
         "Security Standards (restricted profile). Runtime: read-only filesystem, drop "
         "Linux capabilities, use seccomp and AppArmor profiles, network policies for "
         "east-west traffic. Falco for runtime threat detection."
     ), "category": "security", "difficulty": "advanced"},

    # -- CI/CD --
    {"title": "GitHub Actions CI/CD Pipelines",
     "content": (
         "GitHub Actions automates build, test, deploy workflows. Workflow YAML in "
         ".github/workflows/. Triggers: push, pull_request, schedule (cron), "
         "workflow_dispatch (manual). Jobs run on runners (ubuntu-latest, windows-latest). "
         "Steps execute commands or use actions (actions/checkout, actions/setup-python). "
         "Matrix strategy tests across multiple versions. Artifacts upload test results "
         "and build outputs. Secrets managed via repository/environment settings. "
         "Reusable workflows and composite actions reduce duplication. Concurrency "
         "controls prevent parallel deployments."
     ), "category": "cicd", "difficulty": "beginner"},

    {"title": "GitOps with ArgoCD",
     "content": (
         "GitOps uses Git as the single source of truth for infrastructure. ArgoCD "
         "continuously reconciles cluster state with Git repository. Application CRD "
         "defines source (Git repo/path) and destination (cluster/namespace). Sync "
         "policies: auto-sync, self-heal, prune orphaned resources. App of Apps pattern "
         "manages multiple applications. ApplicationSets generate applications from "
         "templates. Health checks: Healthy, Progressing, Degraded, Suspended. "
         "RBAC via SSO integration (OIDC, SAML). Notifications via Slack, email webhooks. "
         "Rollback to any Git commit."
     ), "category": "cicd", "difficulty": "advanced"},

    # -- Databases --
    {"title": "PostgreSQL Performance Tuning",
     "content": (
         "PostgreSQL tuning parameters: shared_buffers (25% of RAM), work_mem (per-sort "
         "memory), effective_cache_size (50-75% of RAM), maintenance_work_mem (256MB-1GB "
         "for vacuum/index). Query optimisation: EXPLAIN ANALYZE, index types (B-tree "
         "default, GIN for full-text/JSONB, GiST for geometric, BRIN for large sorted "
         "tables). Connection pooling: PgBouncer or Pgpool-II. Vacuum: autovacuum "
         "settings, prevent bloat. Partitioning: range, list, or hash for large tables. "
         "pg_stat_statements tracks query performance. Parallel query execution for "
         "large sequential scans."
     ), "category": "databases", "difficulty": "advanced"},

    {"title": "Redis Data Structures and Use Cases",
     "content": (
         "Redis is an in-memory data store supporting: Strings (caching, counters), "
         "Lists (queues, activity feeds), Sets (unique collections, tagging), Sorted "
         "Sets (leaderboards, priority queues), Hashes (objects, session storage), "
         "Streams (event sourcing, message queues), HyperLogLog (cardinality estimation), "
         "Bitmaps (feature flags, real-time analytics). Persistence: RDB snapshots and "
         "AOF append-only file. Clustering: hash slots (16384) distributed across nodes. "
         "Sentinel for high availability. Pub/Sub for real-time messaging. TTL-based "
         "expiry for cache eviction. Lua scripting for atomic operations."
     ), "category": "databases", "difficulty": "intermediate"},

    # -- Observability --
    {"title": "Prometheus Monitoring and Alerting",
     "content": (
         "Prometheus scrapes metrics from /metrics endpoints. PromQL query language: "
         "rate(), histogram_quantile(), increase(), avg_over_time(). Metric types: "
         "Counter (monotonic), Gauge (up/down), Histogram (request latency distribution), "
         "Summary (quantiles). Recording rules pre-compute expensive queries. Alert rules "
         "define conditions and severity. Alertmanager routes alerts to Slack, PagerDuty, "
         "email with deduplication, grouping, and silencing. Service discovery: "
         "kubernetes_sd_configs, consul_sd_configs. Thanos or Cortex for long-term storage."
     ), "category": "observability", "difficulty": "intermediate"},

    {"title": "Distributed Tracing with OpenTelemetry",
     "content": (
         "OpenTelemetry provides vendor-agnostic observability: traces, metrics, logs. "
         "Trace: a DAG of spans representing a request's journey across services. Span "
         "attributes: service.name, http.method, http.status_code, db.statement. Context "
         "propagation: W3C Trace Context (traceparent header). Exporters: Jaeger, Zipkin, "
         "OTLP (to Grafana Tempo, Datadog). Instrumentation: auto-instrumentation for "
         "Python/Java/Go, manual spans for business logic. Sampling strategies: head-based "
         "(probability), tail-based (decision after trace completes). Baggage for "
         "cross-cutting concerns."
     ), "category": "observability", "difficulty": "advanced"},

    # -- Terraform --
    {"title": "Terraform State Management",
     "content": (
         "Terraform state tracks resource mappings. Remote backends: S3+DynamoDB (AWS), "
         "GCS (GCP), Azure Blob. State locking prevents concurrent modifications. "
         "terraform plan shows proposed changes. terraform apply executes them. State "
         "manipulation: terraform state mv, rm, import. Workspaces isolate environments "
         "(dev/staging/prod). State file contains sensitive data -- encrypt backend. "
         "Drift detection: terraform plan shows if real infrastructure diverged. "
         "terraform taint forces resource recreation. Modules encapsulate reusable "
         "infrastructure patterns."
     ), "category": "terraform", "difficulty": "intermediate"},

    {"title": "Terraform Modules and Best Practices",
     "content": (
         "Modules are reusable Terraform configurations with input variables and outputs. "
         "Module structure: main.tf, variables.tf, outputs.tf, versions.tf. Registry "
         "modules from registry.terraform.io. Versioning: source constraints in module "
         "blocks. Best practices: DRY with modules, use locals for computed values, "
         "data sources for existing infrastructure, count/for_each for multiple resources, "
         "lifecycle rules (prevent_destroy, ignore_changes). Terragrunt for multi-account "
         "orchestration. Pre-commit hooks: terraform fmt, validate, tflint, checkov."
     ), "category": "terraform", "difficulty": "advanced"},

    # -- Docker --
    {"title": "Docker Multi-Stage Builds",
     "content": (
         "Multi-stage builds reduce image size by separating build and runtime. Stage 1 "
         "(builder): install dependencies, compile code. Stage 2 (runtime): copy only "
         "artifacts from builder, use minimal base image. Example: Go binary from "
         "golang:1.21 builder to scratch (0 MB base). Python: pip install in builder, "
         "copy site-packages to python:3.11-slim. Layer caching: order Dockerfile "
         "instructions from least to most frequently changing. COPY requirements.txt "
         "before COPY . enables pip cache. BuildKit enables parallel stage execution. "
         ".dockerignore reduces build context size."
     ), "category": "docker", "difficulty": "intermediate"},
]

print(f"Corpus: {len(CORPUS)} documents")
categories = Counter(d["category"] for d in CORPUS)
for cat, n in sorted(categories.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {n} docs")

## Document Store

The Document Store is the persistence layer that connects ingestion to retrieval. In production Haystack, you would use `InMemoryDocumentStore`, `ElasticsearchDocumentStore`, `ChromaDocumentStore`, etc. We build an in-memory store with the same API.

**Key operations:**
- `write_documents()` -- add or update documents
- `filter_documents()` -- metadata-based filtering
- `bm25_retrieval()` -- built-in sparse retrieval for BM25
- `embedding_retrieval()` -- dense vector search

In [ ]:
class InMemoryDocumentStore:
    """Simulates haystack.document_stores.in_memory.InMemoryDocumentStore.

    In production:
        from haystack.document_stores.in_memory import InMemoryDocumentStore
        store = InMemoryDocumentStore()
        store.write_documents(documents)
    """

    def __init__(self):
        self._docs: dict[str, Document] = {}
        self._bm25_index: dict[str, list[str]] = defaultdict(list)  # term -> [doc_ids]
        self._doc_tokens: dict[str, list[str]] = {}
        self._df: Counter = Counter()   # document frequency
        self._avg_dl: float = 0.0       # average document length

    # -- Write operations --
    def write_documents(self, documents: list[Document],
                        policy: str = "overwrite") -> int:
        written = 0
        for doc in documents:
            if policy == "skip" and doc.id in self._docs:
                continue
            self._docs[doc.id] = doc
            tokens = self._tokenize(doc.content)
            self._doc_tokens[doc.id] = tokens
            for t in set(tokens):
                if doc.id not in self._bm25_index[t]:
                    self._bm25_index[t].append(doc.id)
                    self._df[t] += 1
            written += 1

        all_lens = [len(t) for t in self._doc_tokens.values()]
        self._avg_dl = np.mean(all_lens) if all_lens else 0.0
        return written

    def _tokenize(self, text: str) -> list[str]:
        return re.findall(r"[a-z0-9]+", text.lower())

    # -- Read operations --
    def count_documents(self) -> int:
        return len(self._docs)

    def filter_documents(self, filters: dict = None) -> list[Document]:
        results = list(self._docs.values())
        if filters:
            for key, value in filters.items():
                results = [d for d in results
                           if d.meta.get(key) == value]
        return results

    # -- BM25 retrieval (built in to the store) --
    def bm25_retrieval(self, query: str, top_k: int = 5,
                       filters: dict = None,
                       k1: float = 1.5, b: float = 0.75) -> list[Document]:
        query_tokens = self._tokenize(query)
        candidate_ids = set()
        for t in query_tokens:
            candidate_ids.update(self._bm25_index.get(t, []))

        if filters:
            filtered = {d.id for d in self.filter_documents(filters)}
            candidate_ids &= filtered

        n = len(self._docs)
        scores = {}
        for doc_id in candidate_ids:
            doc_tokens = self._doc_tokens[doc_id]
            tf_map = Counter(doc_tokens)
            doc_len = len(doc_tokens)
            score = 0.0
            for qt in query_tokens:
                if qt not in tf_map:
                    continue
                tf = tf_map[qt]
                df = self._df.get(qt, 0)
                idf = math.log((n - df + 0.5) / (df + 0.5) + 1)
                tf_norm = (tf * (k1 + 1)) / (
                    tf + k1 * (1 - b + b * doc_len / max(self._avg_dl, 1))
                )
                score += idf * tf_norm
            scores[doc_id] = score

        ranked = sorted(scores.items(), key=lambda x: -x[1])[:top_k]
        results = []
        for doc_id, score in ranked:
            doc = copy.copy(self._docs[doc_id])
            doc.score = round(score, 4)
            results.append(doc)
        return results

    # -- Embedding retrieval --
    def embedding_retrieval(self, query_embedding: np.ndarray,
                            top_k: int = 5,
                            filters: dict = None) -> list[Document]:
        candidates = self.filter_documents(filters) if filters else list(self._docs.values())
        candidates = [d for d in candidates if d.embedding is not None]

        scores = []
        for doc in candidates:
            sim = float(np.dot(query_embedding, doc.embedding) /
                        (np.linalg.norm(query_embedding) *
                         np.linalg.norm(doc.embedding) + 1e-10))
            scores.append((doc, sim))

        scores.sort(key=lambda x: -x[1])
        results = []
        for doc, sim in scores[:top_k]:
            d = copy.copy(doc)
            d.score = round(sim, 4)
            results.append(d)
        return results


store = InMemoryDocumentStore()
print("InMemoryDocumentStore created.")

## Preprocessing Components

Haystack provides preprocessing components that clean, split, and prepare documents for indexing.

### Component 1: DocumentCleaner
Normalises whitespace, optionally strips HTML, converts to lowercase.

### Component 2: DocumentSplitter
Splits long documents into smaller chunks for more precise retrieval.

In [ ]:
class DocumentCleaner(Component):
    """Normalises document content.

    In production Haystack:
        from haystack.components.preprocessors import DocumentCleaner
        cleaner = DocumentCleaner(remove_empty_lines=True)
    """

    def __init__(self, lowercase: bool = False, strip_whitespace: bool = True):
        self.lowercase = lowercase
        self.strip_whitespace = strip_whitespace

    def run(self, documents: list[Document]) -> dict:
        cleaned = []
        for doc in documents:
            text = doc.content
            if self.strip_whitespace:
                text = re.sub(r"\s+", " ", text).strip()
            if self.lowercase:
                text = text.lower()
            new_doc = Document(id=doc.id, content=text,
                               meta={**doc.meta, "cleaned": True})
            cleaned.append(new_doc)
        return {"documents": cleaned}


class DocumentSplitter(Component):
    """Splits documents into smaller chunks.

    In production Haystack:
        from haystack.components.preprocessors import DocumentSplitter
        splitter = DocumentSplitter(split_by="sentence", split_length=3)
    """

    def __init__(self, split_by: str = "sentence",
                 split_length: int = 2,
                 split_overlap: int = 0):
        self.split_by = split_by
        self.split_length = split_length
        self.split_overlap = split_overlap

    def _split_sentences(self, text: str) -> list[str]:
        return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

    def run(self, documents: list[Document]) -> dict:
        split_docs = []
        for doc in documents:
            if self.split_by == "sentence":
                sentences = self._split_sentences(doc.content)
            else:
                sentences = doc.content.split()

            step = max(1, self.split_length - self.split_overlap)
            for i in range(0, len(sentences), step):
                chunk_sents = sentences[i:i + self.split_length]
                if not chunk_sents:
                    continue
                chunk_text = " ".join(chunk_sents)
                split_doc = Document(
                    content=chunk_text,
                    meta={**doc.meta,
                          "source_id": doc.id,
                          "split_idx": i // step,
                          "split_by": self.split_by},
                )
                split_docs.append(split_doc)
        return {"documents": split_docs}


# Test
cleaner = DocumentCleaner()
splitter = DocumentSplitter(split_by="sentence", split_length=3, split_overlap=1)

sample = [Document(content=CORPUS[0]["content"], meta=CORPUS[0])]
cleaned = cleaner.run(documents=sample)["documents"]
split = splitter.run(documents=cleaned)["documents"]

print(f"Original: 1 doc, {len(sample[0].content)} chars")
print(f"After split: {len(split)} chunks")
for i, s in enumerate(split[:3]):
    print(f"  Chunk {i}: {len(s.content)} chars -- {s.content[:80]}...")

## Embedder Component

Converts text into dense vector representations. In production Haystack, this wraps OpenAI, Sentence-Transformers, or Cohere embedding APIs. Here we simulate with TF-IDF + random projection.

Haystack provides two flavours:
- **DocumentEmbedder** -- embeds document content and stores the vector in `doc.embedding`
- **TextEmbedder** -- embeds a query string and returns the vector

In [ ]:
class SimulatedEmbedder:
    """Shared embedding engine used by both DocumentEmbedder and TextEmbedder."""

    def __init__(self, dim: int = 128):
        self.dim = dim
        self.vocab: dict[str, int] = {}
        self.idf: dict[str, float] = {}
        self._proj: Optional[np.ndarray] = None
        self._fitted = False

    def fit(self, texts: list[str]):
        doc_freq = Counter()
        all_tokens = set()
        for text in texts:
            tokens = set(re.findall(r"[a-z0-9_]+", text.lower()))
            all_tokens.update(tokens)
            for t in tokens:
                doc_freq[t] += 1

        self.vocab = {t: i for i, t in enumerate(sorted(all_tokens))}
        n = len(texts)
        self.idf = {t: math.log(n / (1 + df)) for t, df in doc_freq.items()}
        np.random.seed(42)
        self._proj = np.random.randn(len(self.vocab), self.dim) / math.sqrt(self.dim)
        self._fitted = True

    def embed(self, text: str) -> np.ndarray:
        tokens = re.findall(r"[a-z0-9_]+", text.lower())
        tf = Counter(tokens)
        vec = np.zeros(len(self.vocab))
        for t, cnt in tf.items():
            if t in self.vocab:
                vec[self.vocab[t]] = cnt * self.idf.get(t, 1.0)
        emb = vec @ self._proj
        norm = np.linalg.norm(emb)
        return emb / norm if norm > 0 else emb


class DocumentEmbedder(Component):
    """Embeds documents and stores vectors in doc.embedding.

    In production:
        from haystack.components.embedders import SentenceTransformersDocumentEmbedder
        embedder = SentenceTransformersDocumentEmbedder(model="all-MiniLM-L6-v2")
    """

    def __init__(self, engine: SimulatedEmbedder):
        self._engine = engine

    def run(self, documents: list[Document]) -> dict:
        for doc in documents:
            doc.embedding = self._engine.embed(doc.content)
        return {"documents": documents}


class TextEmbedder(Component):
    """Embeds a query string.

    In production:
        from haystack.components.embedders import SentenceTransformersTextEmbedder
        embedder = SentenceTransformersTextEmbedder(model="all-MiniLM-L6-v2")
    """

    def __init__(self, engine: SimulatedEmbedder):
        self._engine = engine

    def run(self, text: str) -> dict:
        return {"embedding": self._engine.embed(text)}


print("Embedder components defined.")

## Retriever Components

Retrievers are the heart of the search pipeline. We implement three strategies:

| Retriever | Strategy | Best For |
|---|---|---|
| **BM25Retriever** | Sparse lexical matching (TF-IDF style) | Exact keyword lookups, entity names |
| **EmbeddingRetriever** | Dense semantic similarity | Conceptual/paraphrased queries |
| **HybridRetriever** | Combines BM25 + embedding via RRF | Broad coverage across query types |

### BM25 vs Dense: When to Use Which

```
User query: "What port does NodePort expose on?"
  BM25:   Finds "NodePort" exact match  -->  HIGH SCORE
  Dense:  Understands the concept        -->  MEDIUM SCORE

User query: "How can I make my pods survive node failure?"
  BM25:   Keyword overlap is weak        -->  LOW SCORE
  Dense:  Semantic match to Deployments   -->  HIGH SCORE

User query: "PostgreSQL tuning for large tables?"
  BM25:   Matches PostgreSQL, tuning     -->  HIGH SCORE
  Dense:  Understands the concept too    -->  HIGH SCORE
  Hybrid: Best of both worlds            -->  HIGHEST SCORE
```

In [ ]:
class BM25Retriever(Component):
    """Sparse retriever using BM25 scoring.

    In production:
        from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
        retriever = InMemoryBM25Retriever(document_store=store, top_k=5)
    """

    def __init__(self, document_store: InMemoryDocumentStore,
                 top_k: int = 5):
        self.store = document_store
        self.top_k = top_k

    def run(self, query: str, filters: dict = None) -> dict:
        t0 = time.time()
        docs = self.store.bm25_retrieval(query, top_k=self.top_k,
                                          filters=filters)
        elapsed = time.time() - t0
        return {"documents": docs, "latency_ms": elapsed * 1000}


class EmbeddingRetriever(Component):
    """Dense retriever using embedding similarity.

    In production:
        from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
        retriever = InMemoryEmbeddingRetriever(document_store=store, top_k=5)
    """

    def __init__(self, document_store: InMemoryDocumentStore,
                 text_embedder: TextEmbedder,
                 top_k: int = 5):
        self.store = document_store
        self.embedder = text_embedder
        self.top_k = top_k

    def run(self, query: str, filters: dict = None) -> dict:
        t0 = time.time()
        q_emb = self.embedder.run(query)["embedding"]
        docs = self.store.embedding_retrieval(q_emb, top_k=self.top_k,
                                               filters=filters)
        elapsed = time.time() - t0
        return {"documents": docs, "latency_ms": elapsed * 1000}


class HybridRetriever(Component):
    """Combines BM25 and Embedding results via Reciprocal Rank Fusion.

    In production: combine retrievers in a pipeline with a JoinDocuments node.
    """

    def __init__(self, bm25: BM25Retriever, embedding: EmbeddingRetriever,
                 top_k: int = 5, alpha: float = 0.5):
        self.bm25 = bm25
        self.embedding = embedding
        self.top_k = top_k
        self.alpha = alpha  # weight for embedding (1-alpha for BM25)

    def run(self, query: str, filters: dict = None) -> dict:
        t0 = time.time()
        bm25_result = self.bm25.run(query=query, filters=filters)
        emb_result = self.embedding.run(query=query, filters=filters)

        k_rrf = 60  # RRF constant
        rrf_scores: dict[str, float] = {}
        doc_map: dict[str, Document] = {}

        for rank, doc in enumerate(bm25_result["documents"]):
            rrf_scores[doc.id] = rrf_scores.get(doc.id, 0) + \
                (1 - self.alpha) / (k_rrf + rank + 1)
            doc_map[doc.id] = doc

        for rank, doc in enumerate(emb_result["documents"]):
            rrf_scores[doc.id] = rrf_scores.get(doc.id, 0) + \
                self.alpha / (k_rrf + rank + 1)
            if doc.id not in doc_map:
                doc_map[doc.id] = doc

        ranked = sorted(rrf_scores.items(), key=lambda x: -x[1])[:self.top_k]
        results = []
        for doc_id, score in ranked:
            doc = copy.copy(doc_map[doc_id])
            doc.score = round(score, 6)
            results.append(doc)

        elapsed = time.time() - t0
        return {"documents": results, "latency_ms": elapsed * 1000}


print("BM25Retriever, EmbeddingRetriever, HybridRetriever defined.")

## Reader Component (Answer Extraction)

The Reader takes retrieved documents and extracts precise answers. In production Haystack, this is either:
- **ExtractiveReader** -- finds answer spans within documents (e.g. `deepset/roberta-base-squad2`)
- **GenerativeReader** -- uses an LLM to synthesise an answer from context

We implement an extractive reader that uses keyword overlap and sentence matching.

In [ ]:
class ExtractiveReader(Component):
    """Extracts answer spans from retrieved documents.

    In production:
        from haystack.components.readers import ExtractiveReader
        reader = ExtractiveReader(model="deepset/roberta-base-squad2")
    """

    def __init__(self, top_k_answers: int = 3):
        self.top_k = top_k_answers

    def _score_sentence(self, sentence: str, query: str) -> float:
        q_tokens = set(re.findall(r"[a-z0-9]+", query.lower()))
        s_tokens = set(re.findall(r"[a-z0-9]+", sentence.lower()))
        if not q_tokens:
            return 0.0
        overlap = len(q_tokens & s_tokens)
        return overlap / len(q_tokens)

    def run(self, query: str, documents: list[Document]) -> dict:
        candidates = []
        for doc in documents:
            sentences = re.split(r"(?<=[.!?])\s+", doc.content)
            for sent in sentences:
                score = self._score_sentence(sent, query)
                if score > 0:
                    candidates.append({
                        "answer": sent.strip(),
                        "score": round(score, 4),
                        "document_id": doc.id,
                        "context": doc.content[:200],
                        "meta": doc.meta,
                    })

        candidates.sort(key=lambda x: -x["score"])
        top = candidates[:self.top_k]

        return {"answers": top, "documents": documents}


reader = ExtractiveReader(top_k_answers=3)
print("ExtractiveReader defined.")

## Pipeline: Wiring Components Together

A Haystack `Pipeline` is a directed acyclic graph (DAG) of components. Each component's outputs connect to the next component's inputs.

```python
# Production Haystack pipeline:
from haystack import Pipeline

pipe = Pipeline()
pipe.add_component("retriever", InMemoryBM25Retriever(store))
pipe.add_component("reader", ExtractiveReader(model="..."))
pipe.connect("retriever.documents", "reader.documents")

result = pipe.run({"retriever": {"query": "..."}})
```

Our implementation mirrors this API with `add_component()`, `connect()`, and `run()`.

In [ ]:
class Pipeline:
    """Simulates haystack.Pipeline.

    Components are added, connected by name, and executed in topological order.
    """

    def __init__(self, name: str = "pipeline"):
        self.name = name
        self._components: dict[str, Component] = {}
        self._edges: list[tuple[str, str, str, str]] = []  # (src, src_out, dst, dst_in)
        self._execution_order: list[str] = []

    def add_component(self, name: str, component: Component) -> "Pipeline":
        self._components[name] = component
        if name not in self._execution_order:
            self._execution_order.append(name)
        return self

    def connect(self, from_socket: str, to_socket: str) -> "Pipeline":
        src_comp, src_out = from_socket.split(".")
        dst_comp, dst_in = to_socket.split(".")
        self._edges.append((src_comp, src_out, dst_comp, dst_in))
        return self

    def run(self, data: dict) -> dict:
        """Execute the pipeline.

        data: dict mapping component names to their initial kwargs.
              Example: {"retriever": {"query": "What is a Pod?"}}
        """
        outputs: dict[str, dict] = {}
        trace: list[dict] = []

        for comp_name in self._execution_order:
            comp = self._components[comp_name]

            # Gather inputs from data and previous outputs
            kwargs = dict(data.get(comp_name, {}))
            for src, src_out, dst, dst_in in self._edges:
                if dst == comp_name and src in outputs:
                    kwargs[dst_in] = outputs[src].get(src_out)

            t0 = time.time()
            result = comp.run(**kwargs)
            elapsed = (time.time() - t0) * 1000
            outputs[comp_name] = result

            trace.append({
                "component": comp_name,
                "type": comp.component_name,
                "duration_ms": round(elapsed, 2),
                "output_keys": list(result.keys()),
            })

        return {"outputs": outputs, "trace": trace}

    def show(self):
        print(f"Pipeline: {self.name}")
        print(f"  Components ({len(self._components)}):")
        for name, comp in self._components.items():
            print(f"    {name}: {comp.component_name}")
        print(f"  Connections ({len(self._edges)}):")
        for src, so, dst, di in self._edges:
            print(f"    {src}.{so} --> {dst}.{di}")


print("Pipeline class defined.")

## Indexing Pipeline

Ingests raw corpus documents through cleaning, splitting, embedding, and writes to the document store.

```
[Documents] --> [Cleaner] --> [Splitter] --> [Embedder] --> [DocumentStore]
```

In [ ]:
# 1. Convert corpus to Documents
raw_documents = [
    Document(content=d["content"],
             meta={"title": d["title"], "category": d["category"],
                   "difficulty": d["difficulty"]})
    for d in CORPUS
]
print(f"Raw documents: {len(raw_documents)}")

# 2. Clean
cleaner = DocumentCleaner(strip_whitespace=True)
cleaned = cleaner.run(documents=raw_documents)["documents"]
print(f"Cleaned: {len(cleaned)}")

# 3. Split into chunks (3 sentences per chunk, 1 sentence overlap)
splitter = DocumentSplitter(split_by="sentence", split_length=3, split_overlap=1)
chunks = splitter.run(documents=cleaned)["documents"]
print(f"Chunks: {len(chunks)}")

# 4. Fit embedder on all chunk content
embed_engine = SimulatedEmbedder(dim=128)
embed_engine.fit([c.content for c in chunks])
print(f"Embedder fitted: vocab={len(embed_engine.vocab)}, dim={embed_engine.dim}")

# 5. Embed all chunks
doc_embedder = DocumentEmbedder(embed_engine)
embedded = doc_embedder.run(documents=chunks)["documents"]
print(f"Embedded: {len(embedded)} chunks ({embed_engine.dim}-dim vectors)")

# 6. Write to document store
n_written = store.write_documents(embedded)
print(f"\nDocument store: {store.count_documents()} documents written")
print(f"Average chunk length: {np.mean([d.content_length for d in embedded]):.0f} chars")

## Query Pipelines

We build three query pipelines, one per retriever, each connected to the same Reader.

In [ ]:
# Create retrievers
text_embedder = TextEmbedder(embed_engine)
bm25_retriever = BM25Retriever(store, top_k=5)
emb_retriever = EmbeddingRetriever(store, text_embedder, top_k=5)
hybrid_retriever = HybridRetriever(bm25_retriever, emb_retriever, top_k=5)

# -- BM25 Pipeline --
bm25_pipe = Pipeline("bm25_qa")
bm25_pipe.add_component("retriever", bm25_retriever)
bm25_pipe.add_component("reader", ExtractiveReader(top_k_answers=3))
bm25_pipe.connect("retriever.documents", "reader.documents")

# -- Embedding Pipeline --
emb_pipe = Pipeline("embedding_qa")
emb_pipe.add_component("retriever", emb_retriever)
emb_pipe.add_component("reader", ExtractiveReader(top_k_answers=3))
emb_pipe.connect("retriever.documents", "reader.documents")

# -- Hybrid Pipeline --
hybrid_pipe = Pipeline("hybrid_qa")
hybrid_pipe.add_component("retriever", hybrid_retriever)
hybrid_pipe.add_component("reader", ExtractiveReader(top_k_answers=3))
hybrid_pipe.connect("retriever.documents", "reader.documents")

print("Query pipelines built:")
for pipe in [bm25_pipe, emb_pipe, hybrid_pipe]:
    pipe.show()
    print()

## Demo: Running a Query Pipeline

In [ ]:
demo_query = "How does Kubernetes handle pod failures and restarts?"

print(f"Query: {demo_query}")
print("=" * 70)

for pipe_name, pipe in [("BM25", bm25_pipe), ("Embedding", emb_pipe),
                         ("Hybrid", hybrid_pipe)]:
    result = pipe.run({
        "retriever": {"query": demo_query},
        "reader": {"query": demo_query},
    })

    docs = result["outputs"]["retriever"]["documents"]
    answers = result["outputs"]["reader"]["answers"]
    trace = result["trace"]

    print(f"\n--- {pipe_name} Pipeline ---")
    print(f"Retrieved {len(docs)} documents:")
    for i, d in enumerate(docs, 1):
        print(f"  {i}. (score={d.score:.4f}) [{d.meta.get('category','')}] "
              f"{d.content[:80]}...")

    if answers:
        print(f"\nTop answer (confidence={answers[0]['score']:.2f}):")
        print(f"  {answers[0]['answer'][:120]}")

    total_ms = sum(t["duration_ms"] for t in trace)
    print(f"Latency: {total_ms:.1f}ms")

## Evaluation Framework

Haystack provides built-in evaluation metrics through the `EvaluationRunResult` class. We implement the same metrics manually:

### Retrieval Metrics
| Metric | Definition | Range |
|---|---|---|
| **Recall@K** | Fraction of relevant docs in top-K results | 0 -- 1 |
| **MRR** (Mean Reciprocal Rank) | 1/rank of first relevant result, averaged | 0 -- 1 |
| **nDCG@K** | Normalised discounted cumulative gain | 0 -- 1 |
| **Precision@K** | Fraction of top-K results that are relevant | 0 -- 1 |

### Answer Metrics
| Metric | Definition | Range |
|---|---|---|
| **Exact Match** | Answer exactly matches ground truth | 0 or 1 |
| **Token F1** | Harmonic mean of token precision and recall | 0 -- 1 |

In [ ]:
def recall_at_k(retrieved_ids: list[str], relevant_ids: set[str],
                k: int = 5) -> float:
    top_k = set(retrieved_ids[:k])
    if not relevant_ids:
        return 0.0
    return len(top_k & relevant_ids) / len(relevant_ids)


def precision_at_k(retrieved_ids: list[str], relevant_ids: set[str],
                   k: int = 5) -> float:
    top_k = retrieved_ids[:k]
    if not top_k:
        return 0.0
    return sum(1 for did in top_k if did in relevant_ids) / len(top_k)


def mrr(retrieved_ids: list[str], relevant_ids: set[str]) -> float:
    for rank, doc_id in enumerate(retrieved_ids, 1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved_ids: list[str], relevant_ids: set[str],
              k: int = 5) -> float:
    def dcg(ids: list[str]) -> float:
        return sum(
            (1.0 if did in relevant_ids else 0.0) / math.log2(i + 2)
            for i, did in enumerate(ids[:k])
        )
    actual = dcg(retrieved_ids)
    ideal_list = ([did for did in retrieved_ids if did in relevant_ids] +
                  [did for did in retrieved_ids if did not in relevant_ids])
    ideal = dcg(ideal_list[:k])
    return actual / ideal if ideal > 0 else 0.0


def token_f1(prediction: str, ground_truth: str) -> float:
    pred_tokens = set(re.findall(r"[a-z0-9]+", prediction.lower()))
    gt_tokens = set(re.findall(r"[a-z0-9]+", ground_truth.lower()))
    if not pred_tokens or not gt_tokens:
        return 0.0
    common = pred_tokens & gt_tokens
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


print("Evaluation metrics defined: recall@k, precision@k, MRR, nDCG@k, token_f1")

## Evaluation Dataset: 30 Labelled Queries

Each query has:
- **Relevant document titles** (ground truth)
- **Expected answer** snippet
- **Query type**: factual, conceptual, procedural, or comparison
- **Difficulty**: easy, medium, hard

In [ ]:
EVAL_QUERIES = [
    # Kubernetes
    {"query": "What are the phases of a Kubernetes Pod?",
     "relevant_titles": ["Kubernetes Pod Lifecycle"],
     "expected_answer": "Pending, Running, Succeeded, Failed, and Unknown",
     "query_type": "factual", "difficulty": "easy"},
    {"query": "How do liveness probes work in Kubernetes?",
     "relevant_titles": ["Kubernetes Pod Lifecycle"],
     "expected_answer": "HTTP, TCP, or exec checks",
     "query_type": "factual", "difficulty": "easy"},
    {"query": "What is the difference between ClusterIP and NodePort?",
     "relevant_titles": ["Kubernetes Services and Networking"],
     "expected_answer": "ClusterIP exposes on internal IP, NodePort on each node IP at static port",
     "query_type": "comparison", "difficulty": "medium"},
    {"query": "How does Horizontal Pod Autoscaler scale deployments?",
     "relevant_titles": ["Kubernetes Deployments and Scaling"],
     "expected_answer": "scales replicas based on CPU/memory or custom metrics",
     "query_type": "conceptual", "difficulty": "medium"},
    {"query": "How should Kubernetes secrets be managed securely?",
     "relevant_titles": ["Kubernetes ConfigMaps and Secrets"],
     "expected_answer": "external secret managers like Vault, never commit to Git",
     "query_type": "procedural", "difficulty": "medium"},
    {"query": "What is Helm and what does it do?",
     "relevant_titles": ["Helm Charts and Package Management"],
     "expected_answer": "package manager for Kubernetes, charts are YAML template bundles",
     "query_type": "factual", "difficulty": "easy"},

    # AWS
    {"query": "What are the EC2 pricing models?",
     "relevant_titles": ["AWS EC2 Instance Types and Pricing"],
     "expected_answer": "On-Demand, Reserved, Spot, and Savings Plans",
     "query_type": "factual", "difficulty": "easy"},
    {"query": "What is the principle behind IAM security?",
     "relevant_titles": ["AWS IAM Policies and Security"],
     "expected_answer": "least privilege principle, use roles instead of long-term access keys",
     "query_type": "conceptual", "difficulty": "medium"},
    {"query": "How does VPC networking work in AWS?",
     "relevant_titles": ["AWS VPC and Network Architecture"],
     "expected_answer": "isolated network with public and private subnets, security groups, NACLs",
     "query_type": "conceptual", "difficulty": "medium"},
    {"query": "What are Lambda cold starts and how to avoid them?",
     "relevant_titles": ["AWS Lambda and Serverless Computing"],
     "expected_answer": "add latency 100ms-10s, use provisioned concurrency to eliminate",
     "query_type": "procedural", "difficulty": "hard"},
    {"query": "Compare RDS Multi-AZ and read replicas",
     "relevant_titles": ["AWS RDS and Database Services"],
     "expected_answer": "Multi-AZ for failover, read replicas for read-heavy workloads",
     "query_type": "comparison", "difficulty": "medium"},

    # Networking
    {"query": "What DNS record types exist?",
     "relevant_titles": ["DNS Resolution and Record Types"],
     "expected_answer": "A, AAAA, CNAME, MX, TXT, NS, SOA, SRV, CAA",
     "query_type": "factual", "difficulty": "easy"},
    {"query": "How does TLS 1.3 handshake work?",
     "relevant_titles": ["TLS/SSL Certificates and HTTPS"],
     "expected_answer": "ClientHello, ServerHello, certificate exchange, key agreement",
     "query_type": "procedural", "difficulty": "hard"},
    {"query": "What load balancing algorithms are available?",
     "relevant_titles": ["Load Balancing Strategies"],
     "expected_answer": "round-robin, least connections, IP hash, weighted round-robin",
     "query_type": "factual", "difficulty": "easy"},

    # Security
    {"query": "What are the OWASP Top 10 risks?",
     "relevant_titles": ["OWASP Top 10 Web Security Risks"],
     "expected_answer": "Broken Access Control, Cryptographic Failures, Injection",
     "query_type": "factual", "difficulty": "easy"},
    {"query": "What is Zero Trust architecture?",
     "relevant_titles": ["Zero Trust Architecture"],
     "expected_answer": "verify every request regardless of network location, assume breach",
     "query_type": "conceptual", "difficulty": "medium"},
    {"query": "How to secure containers at build time and runtime?",
     "relevant_titles": ["Container Security Best Practices"],
     "expected_answer": "minimal base images, scan CVEs, read-only filesystem, drop capabilities",
     "query_type": "procedural", "difficulty": "hard"},

    # CI/CD
    {"query": "How do GitHub Actions workflows trigger?",
     "relevant_titles": ["GitHub Actions CI/CD Pipelines"],
     "expected_answer": "push, pull_request, schedule, workflow_dispatch",
     "query_type": "factual", "difficulty": "easy"},
    {"query": "What is GitOps and how does ArgoCD implement it?",
     "relevant_titles": ["GitOps with ArgoCD"],
     "expected_answer": "Git as single source of truth, ArgoCD reconciles cluster state",
     "query_type": "conceptual", "difficulty": "medium"},

    # Databases
    {"query": "What PostgreSQL parameters should be tuned for performance?",
     "relevant_titles": ["PostgreSQL Performance Tuning"],
     "expected_answer": "shared_buffers 25% RAM, work_mem, effective_cache_size",
     "query_type": "procedural", "difficulty": "hard"},
    {"query": "What Redis data structures are available?",
     "relevant_titles": ["Redis Data Structures and Use Cases"],
     "expected_answer": "Strings, Lists, Sets, Sorted Sets, Hashes, Streams, HyperLogLog",
     "query_type": "factual", "difficulty": "easy"},

    # Observability
    {"query": "What are Prometheus metric types?",
     "relevant_titles": ["Prometheus Monitoring and Alerting"],
     "expected_answer": "Counter, Gauge, Histogram, Summary",
     "query_type": "factual", "difficulty": "easy"},
    {"query": "How does distributed tracing propagate context?",
     "relevant_titles": ["Distributed Tracing with OpenTelemetry"],
     "expected_answer": "W3C Trace Context via traceparent header",
     "query_type": "conceptual", "difficulty": "hard"},

    # Terraform
    {"query": "How does Terraform manage state?",
     "relevant_titles": ["Terraform State Management"],
     "expected_answer": "remote backends S3 DynamoDB, state locking prevents concurrent modifications",
     "query_type": "conceptual", "difficulty": "medium"},
    {"query": "What are Terraform module best practices?",
     "relevant_titles": ["Terraform Modules and Best Practices"],
     "expected_answer": "DRY with modules, use locals, data sources for existing infrastructure",
     "query_type": "procedural", "difficulty": "medium"},

    # Docker
    {"query": "How do multi-stage Docker builds reduce image size?",
     "relevant_titles": ["Docker Multi-Stage Builds"],
     "expected_answer": "separate build and runtime, copy only artifacts to minimal base image",
     "query_type": "conceptual", "difficulty": "medium"},

    # Cross-topic / harder
    {"query": "How can I make my application resilient to infrastructure failures?",
     "relevant_titles": ["Kubernetes Deployments and Scaling", "AWS RDS and Database Services",
                          "Load Balancing Strategies"],
     "expected_answer": "autoscaling, multi-AZ, load balancing, rolling updates",
     "query_type": "conceptual", "difficulty": "hard"},
    {"query": "What security measures protect data in transit and at rest?",
     "relevant_titles": ["TLS/SSL Certificates and HTTPS", "AWS IAM Policies and Security",
                          "AWS RDS and Database Services"],
     "expected_answer": "TLS 1.3, KMS encryption, HSTS, certificate management",
     "query_type": "conceptual", "difficulty": "hard"},
    {"query": "How do I set up monitoring and alerting for a production system?",
     "relevant_titles": ["Prometheus Monitoring and Alerting",
                          "Distributed Tracing with OpenTelemetry"],
     "expected_answer": "Prometheus scrapes metrics, AlertManager routes alerts, OpenTelemetry tracing",
     "query_type": "procedural", "difficulty": "hard"},
    {"query": "Compare serverless vs container deployment approaches",
     "relevant_titles": ["AWS Lambda and Serverless Computing",
                          "Docker Multi-Stage Builds",
                          "Kubernetes Deployments and Scaling"],
     "expected_answer": "Lambda for event-driven, containers for long-running, Kubernetes for orchestration",
     "query_type": "comparison", "difficulty": "hard"},
]

print(f"Evaluation dataset: {len(EVAL_QUERIES)} queries")
query_types = Counter(q["query_type"] for q in EVAL_QUERIES)
difficulties = Counter(q["difficulty"] for q in EVAL_QUERIES)
print(f"  Types: {dict(query_types)}")
print(f"  Difficulty: {dict(difficulties)}")

## Mapping Ground Truth to Chunk IDs

Our retrieval evaluation compares retrieved chunk IDs against the IDs of chunks that originated from relevant documents.

In [ ]:
# Build title -> chunk IDs mapping
title_to_chunk_ids: dict[str, set[str]] = defaultdict(set)
for doc in store.filter_documents():
    title = doc.meta.get("title", "")
    if title:
        title_to_chunk_ids[title].add(doc.id)

# Build evaluation ground truth
eval_data = []
for q in EVAL_QUERIES:
    relevant_ids = set()
    for title in q["relevant_titles"]:
        relevant_ids.update(title_to_chunk_ids.get(title, set()))
    eval_data.append({
        "query": q["query"],
        "relevant_ids": relevant_ids,
        "expected_answer": q["expected_answer"],
        "query_type": q["query_type"],
        "difficulty": q["difficulty"],
    })

# Sanity check
matched = sum(1 for e in eval_data if len(e["relevant_ids"]) > 0)
print(f"Queries with matched ground truth: {matched}/{len(eval_data)}")
print(f"Total relevant chunks across all queries: "
      f"{sum(len(e['relevant_ids']) for e in eval_data)}")

## Running the Full Evaluation

We evaluate each retriever pipeline on all 30 queries and collect retrieval + answer metrics.

In [ ]:
K = 5  # evaluation at K=5

pipelines = {
    "BM25": bm25_pipe,
    "Embedding": emb_pipe,
    "Hybrid": hybrid_pipe,
}

all_results = []

for pipe_name, pipe in pipelines.items():
    for ed in eval_data:
        result = pipe.run({
            "retriever": {"query": ed["query"]},
            "reader": {"query": ed["query"]},
        })

        ret_docs = result["outputs"]["retriever"]["documents"]
        answers = result["outputs"]["reader"]["answers"]
        ret_ids = [d.id for d in ret_docs]

        # Retrieval metrics
        r_at_k = recall_at_k(ret_ids, ed["relevant_ids"], k=K)
        p_at_k = precision_at_k(ret_ids, ed["relevant_ids"], k=K)
        m = mrr(ret_ids, ed["relevant_ids"])
        n = ndcg_at_k(ret_ids, ed["relevant_ids"], k=K)

        # Answer metrics
        top_answer = answers[0]["answer"] if answers else ""
        f1 = token_f1(top_answer, ed["expected_answer"])

        latency = sum(t["duration_ms"] for t in result["trace"])

        all_results.append({
            "pipeline": pipe_name,
            "query": ed["query"][:60],
            "query_type": ed["query_type"],
            "difficulty": ed["difficulty"],
            "recall@5": r_at_k,
            "precision@5": p_at_k,
            "mrr": m,
            "ndcg@5": n,
            "answer_f1": f1,
            "latency_ms": round(latency, 2),
            "n_relevant": len(ed["relevant_ids"]),
        })

results_df = pd.DataFrame(all_results)
print(f"Evaluation complete: {len(all_results)} pipeline-query pairs")
print(f"  Pipelines: {list(pipelines.keys())}")
print(f"  Queries: {len(eval_data)}")

## Results: Overall Retriever Comparison

In [ ]:
overall = results_df.groupby("pipeline").agg(
    recall_5=("recall@5", "mean"),
    precision_5=("precision@5", "mean"),
    mrr=("mrr", "mean"),
    ndcg_5=("ndcg@5", "mean"),
    answer_f1=("answer_f1", "mean"),
    latency_ms=("latency_ms", "mean"),
).round(4)

print("OVERALL RETRIEVER COMPARISON")
print("=" * 80)
print(overall.to_string())

best_recall = overall["recall_5"].idxmax()
best_mrr = overall["mrr"].idxmax()
best_f1 = overall["answer_f1"].idxmax()
print(f"\nBest Recall@5:  {best_recall} ({overall.loc[best_recall, 'recall_5']:.4f})")
print(f"Best MRR:       {best_mrr} ({overall.loc[best_mrr, 'mrr']:.4f})")
print(f"Best Answer F1: {best_f1} ({overall.loc[best_f1, 'answer_f1']:.4f})")

In [ ]:
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["Recall@5", "MRR", "Answer F1"])

colors = {"BM25": "#3498db", "Embedding": "#e74c3c", "Hybrid": "#2ecc71"}

for metric, col in [("recall_5", 1), ("mrr", 2), ("answer_f1", 3)]:
    for pipeline in overall.index:
        fig.add_trace(go.Bar(
            x=[pipeline], y=[overall.loc[pipeline, metric]],
            marker_color=colors[pipeline],
            name=pipeline, showlegend=(col == 1),
        ), row=1, col=col)

fig.update_layout(
    height=400, template="plotly_white",
    title_text="Retriever Performance Comparison",
    barmode="group",
)
fig.update_yaxes(range=[0, 1])
fig.show()

## Results: Performance by Query Type

Different retriever strategies excel at different query types.

In [ ]:
by_type = results_df.groupby(["pipeline", "query_type"]).agg(
    recall_5=("recall@5", "mean"),
    mrr=("mrr", "mean"),
    answer_f1=("answer_f1", "mean"),
).round(4).unstack("pipeline")

print("PERFORMANCE BY QUERY TYPE")
print("=" * 80)
print(by_type.to_string())

In [ ]:
fig = px.imshow(
    results_df.pivot_table(
        index="query_type", columns="pipeline",
        values="recall@5", aggfunc="mean"
    ).values,
    x=list(pipelines.keys()),
    y=sorted(results_df["query_type"].unique()),
    color_continuous_scale="YlGnBu",
    text_auto=".3f",
    title="Recall@5: Query Type x Retriever",
    labels=dict(x="Retriever", y="Query Type", color="Recall@5"),
)
fig.update_layout(template="plotly_white", height=350)
fig.show()

## Results: Performance by Difficulty Level

In [ ]:
by_diff = results_df.groupby(["pipeline", "difficulty"]).agg(
    recall_5=("recall@5", "mean"),
    mrr=("mrr", "mean"),
    answer_f1=("answer_f1", "mean"),
).round(4)

print("PERFORMANCE BY DIFFICULTY")
print("=" * 80)
print(by_diff.unstack("pipeline").to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
diff_order = ["easy", "medium", "hard"]

for ax, metric, title in zip(axes,
                              ["recall@5", "mrr", "answer_f1"],
                              ["Recall@5", "MRR", "Answer F1"]):
    for pipe_name in pipelines:
        vals = [by_diff.loc[(pipe_name, d), metric]
                if (pipe_name, d) in by_diff.index else 0
                for d in diff_order]
        ax.plot(diff_order, vals, "o-", label=pipe_name,
                color=colors[pipe_name], linewidth=2, markersize=8)
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.set_xlabel("Difficulty")

plt.tight_layout()
plt.show()

## Radar Chart: Multi-Dimensional Comparison

In [ ]:
radar_metrics = ["recall_5", "precision_5", "mrr", "ndcg_5", "answer_f1"]
radar_labels = ["Recall@5", "Precision@5", "MRR", "nDCG@5", "Answer F1"]

fig = go.Figure()
for pipe_name in pipelines:
    vals = [overall.loc[pipe_name, m] for m in radar_metrics]
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=radar_labels + [radar_labels[0]],
        name=pipe_name, fill="toself",
        line=dict(color=colors[pipe_name]),
        opacity=0.6,
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="Retriever Quality Radar",
    template="plotly_white", height=450,
)
fig.show()

## Latency Analysis

In [ ]:
fig = px.box(results_df, x="pipeline", y="latency_ms",
             color="pipeline", color_discrete_map=colors,
             title="Query Latency Distribution by Retriever",
             template="plotly_white")
fig.update_layout(height=400, showlegend=False)
fig.show()

latency_summary = results_df.groupby("pipeline")["latency_ms"].agg(
    ["mean", "median", "std", "min", "max"]
).round(2)
print("Latency Summary (ms):")
print(latency_summary.to_string())

## Per-Query Spotlight

Detailed look at specific queries where retrievers disagree.

In [ ]:
# Find queries with biggest recall gap between retrievers
pivot = results_df.pivot_table(
    index="query", columns="pipeline", values="recall@5"
)
pivot["gap"] = pivot.max(axis=1) - pivot.min(axis=1)
top_gaps = pivot.nlargest(5, "gap")

print("TOP 5 QUERIES WITH BIGGEST RETRIEVER DISAGREEMENT")
print("=" * 80)
for query, row in top_gaps.iterrows():
    print(f"\n  Query: {query}")
    for pipe in pipelines:
        r = row.get(pipe, 0)
        marker = " <--BEST" if r == row[list(pipelines.keys())].max() else ""
        print(f"    {pipe:12s}: recall@5 = {r:.3f}{marker}")
    print(f"    Gap: {row['gap']:.3f}")

## Advanced: Building Custom Components

One of Haystack's strengths is extensibility. Any class that implements `run()` can be a component. Here we build a `QueryClassifier` that routes queries to the best retriever based on query characteristics.

```python
# In production Haystack:
@component
class QueryClassifier:
    @component.output_types(
        keyword_query=str, semantic_query=str
    )
    def run(self, query: str) -> dict:
        if self._is_keyword_query(query):
            return {"keyword_query": query}
        return {"semantic_query": query}
```

In [ ]:
class QueryClassifier(Component):
    """Routes queries to the optimal retriever based on query characteristics.

    Classification rules:
    - Factual/entity queries -> BM25 (exact keyword matches)
    - Conceptual/paraphrased -> Embedding (semantic similarity)
    - Complex/multi-topic    -> Hybrid (coverage across methods)
    """

    KEYWORD_PATTERNS = [
        r"\bwhat (?:are|is) the\b",          # definitional
        r"\blist\b|\bhow many\b",             # enumeration
        r"\bwhat .* types?\b",                # type listing
        r"\bwhat .* (port|version|limit)\b",  # specific values
    ]

    CONCEPTUAL_PATTERNS = [
        r"\bhow does .* work\b",              # mechanism
        r"\bwhy\b",                           # reasoning
        r"\bexplain\b|\bdescribe\b",          # explanation
        r"\bwhat is (?:the )?(?:difference|concept)\b",
    ]

    def run(self, query: str) -> dict:
        q_lower = query.lower()

        for pat in self.KEYWORD_PATTERNS:
            if re.search(pat, q_lower):
                return {"route": "bm25", "query": query,
                        "reason": f"Matched keyword pattern: {pat}"}

        for pat in self.CONCEPTUAL_PATTERNS:
            if re.search(pat, q_lower):
                return {"route": "embedding", "query": query,
                        "reason": f"Matched conceptual pattern: {pat}"}

        # Default: hybrid for complex/ambiguous queries
        return {"route": "hybrid", "query": query,
                "reason": "No strong signal -- defaulting to hybrid"}


classifier = QueryClassifier()

# Test classification
test_queries = [
    "What are the phases of a Kubernetes Pod?",
    "How does zero trust architecture work?",
    "Compare Lambda and containers for ML deployment",
    "What DNS record types exist?",
]

print("QUERY CLASSIFICATION RESULTS:")
print("-" * 65)
for q in test_queries:
    result = classifier.run(query=q)
    print(f"  [{result['route']:>10s}] {q}")
    print(f"             Reason: {result['reason'][:60]}")

## Adaptive Pipeline with Query Routing

Combine the classifier with per-type retrievers for optimal routing.

In [ ]:
class AdaptivePipeline:
    """Routes each query to the best retriever based on classification."""

    def __init__(self, classifier: QueryClassifier,
                 pipelines: dict[str, Pipeline]):
        self.classifier = classifier
        self.pipelines = pipelines   # route_name -> Pipeline

    def run(self, query: str) -> dict:
        classification = self.classifier.run(query=query)
        route = classification["route"]

        # Fall back to hybrid if route not available
        pipe_name = route if route in self.pipelines else "hybrid"
        pipe = self.pipelines.get(pipe_name, list(self.pipelines.values())[0])

        result = pipe.run({
            "retriever": {"query": query},
            "reader": {"query": query},
        })
        result["classification"] = classification
        result["selected_pipeline"] = pipe_name
        return result


adaptive = AdaptivePipeline(
    classifier=classifier,
    pipelines={"bm25": bm25_pipe, "embedding": emb_pipe, "hybrid": hybrid_pipe},
)

# Evaluate adaptive pipeline
adaptive_results = []
for ed in eval_data:
    result = adaptive.run(query=ed["query"])
    ret_docs = result["outputs"]["retriever"]["documents"]
    answers = result["outputs"]["reader"]["answers"]
    ret_ids = [d.id for d in ret_docs]

    r_at_k = recall_at_k(ret_ids, ed["relevant_ids"], k=K)
    m = mrr(ret_ids, ed["relevant_ids"])
    top_answer = answers[0]["answer"] if answers else ""
    f1 = token_f1(top_answer, ed["expected_answer"])

    adaptive_results.append({
        "pipeline": "Adaptive",
        "route": result["classification"]["route"],
        "query": ed["query"][:60],
        "query_type": ed["query_type"],
        "difficulty": ed["difficulty"],
        "recall@5": r_at_k, "mrr": m, "answer_f1": f1,
    })

adaptive_df = pd.DataFrame(adaptive_results)

# Compare against fixed pipelines
print("ADAPTIVE PIPELINE RESULTS")
print("=" * 60)
adaptive_overall = adaptive_df[["recall@5", "mrr", "answer_f1"]].mean()
print(f"  Recall@5:   {adaptive_overall['recall@5']:.4f}")
print(f"  MRR:        {adaptive_overall['mrr']:.4f}")
print(f"  Answer F1:  {adaptive_overall['answer_f1']:.4f}")

print(f"\nRouting distribution:")
print(adaptive_df["route"].value_counts().to_string())

print(f"\nvs Fixed Pipelines:")
for pipe_name in pipelines:
    row = overall.loc[pipe_name]
    print(f"  {pipe_name:12s}: recall={row['recall_5']:.4f}  "
          f"mrr={row['mrr']:.4f}  f1={row['answer_f1']:.4f}")
print(f"  {'Adaptive':12s}: recall={adaptive_overall['recall@5']:.4f}  "
      f"mrr={adaptive_overall['mrr']:.4f}  "
      f"f1={adaptive_overall['answer_f1']:.4f}")

## Source Category Heatmap

Which document categories are retrieved for which query types?

In [ ]:
# For each query, check what categories appear in BM25 top-5
cat_query_matrix = defaultdict(lambda: defaultdict(int))

for ed in eval_data:
    result = bm25_pipe.run({
        "retriever": {"query": ed["query"]},
        "reader": {"query": ed["query"]},
    })
    for doc in result["outputs"]["retriever"]["documents"]:
        cat = doc.meta.get("category", "unknown")
        cat_query_matrix[ed["query_type"]][cat] += 1

# Build DataFrame
cats = sorted({c for row in cat_query_matrix.values() for c in row})
qtypes = sorted(cat_query_matrix.keys())
matrix = pd.DataFrame(
    [[cat_query_matrix[qt].get(c, 0) for c in cats] for qt in qtypes],
    index=qtypes, columns=cats,
)

fig = px.imshow(
    matrix.values,
    x=matrix.columns.tolist(),
    y=matrix.index.tolist(),
    color_continuous_scale="Oranges",
    text_auto=True,
    title="Retrieved Document Categories by Query Type (BM25)",
    labels=dict(x="Document Category", y="Query Type", color="Count"),
)
fig.update_layout(template="plotly_white", height=350)
fig.show()

## Pipeline Serialisation

Production Haystack pipelines can be serialised to YAML for version control and reproducibility:

```yaml
# pipeline.yaml
components:
  retriever:
    type: InMemoryBM25Retriever
    init_parameters:
      document_store: InMemoryDocumentStore
      top_k: 5
  reader:
    type: ExtractiveReader
    init_parameters:
      model: deepset/roberta-base-squad2
      top_k: 3

connections:
  - sender: retriever.documents
    receiver: reader.documents
```

Our pipeline's `show()` method mirrors this structure:

In [ ]:
print("Pipeline configurations:")
print()
for name, pipe in [("BM25 QA", bm25_pipe), ("Embedding QA", emb_pipe),
                    ("Hybrid QA", hybrid_pipe)]:
    print(f"--- {name} ---")
    pipe.show()
    print()

## When to Use Haystack

### Haystack vs Alternatives

| Feature | Haystack | LlamaIndex | LangChain |
|---|---|---|---|
| **Primary focus** | Production search + QA | Data connectors + RAG | LLM app framework |
| **Component protocol** | Strict typed I/O | Flexible | Chain/Agent based |
| **Evaluation built-in** | Yes (comprehensive) | Basic metrics | Community tools |
| **Pipeline DAG** | First-class citizen | Workflow (newer) | LCEL chains |
| **Document stores** | Many native adapters | VectorStoreIndex | Vectorstore wrapper |
| **Best for** | Enterprise search, QA systems | Multi-source RAG | General LLM apps |

### Haystack Design Principles
1. **Pipeline-first**: Everything is a pipeline -- indexing, querying, evaluation
2. **Type safety**: Component inputs/outputs are validated at pipeline build time
3. **Swappability**: Change retriever or LLM by changing one component
4. **Evaluation-driven**: Measure before optimising
5. **Production-ready**: REST API, telemetry, serialisation from day one

## Evaluation Summary

In [ ]:
print("HAYSTACK SEARCH & ANSWER PIPELINE -- EVALUATION SUMMARY")
print("=" * 70)

print(f"\nCorpus: {len(CORPUS)} source documents -> {store.count_documents()} indexed chunks")
print(f"Evaluation: {len(EVAL_QUERIES)} queries, K={K}")

print(f"\n--- Retriever Ranking (by MRR) ---")
ranked = overall.sort_values("mrr", ascending=False)
for i, (name, row) in enumerate(ranked.iterrows(), 1):
    print(f"  {i}. {name:12s}: MRR={row['mrr']:.4f}  "
          f"Recall@5={row['recall_5']:.4f}  nDCG@5={row['ndcg_5']:.4f}  "
          f"F1={row['answer_f1']:.4f}")

print(f"\n--- Adaptive Pipeline ---")
print(f"  Recall@5={adaptive_overall['recall@5']:.4f}  "
      f"MRR={adaptive_overall['mrr']:.4f}  "
      f"F1={adaptive_overall['answer_f1']:.4f}")

print(f"\n--- Recommendations ---")
print(f"  Factual/keyword queries:   Use BM25  (fast, exact match)")
print(f"  Conceptual queries:        Use Embedding (semantic understanding)")
print(f"  Mixed/unknown queries:     Use Hybrid or Adaptive routing")
print(f"  Production deployment:     Use Adaptive with QueryClassifier")

## Common Mistakes

1. **Using only BM25**: Misses paraphrased and conceptual queries. Always benchmark against dense retrieval.
2. **Not splitting documents**: Long documents dilute relevance scores. Chunk to 2-3 sentences for precise retrieval.
3. **Ignoring evaluation**: "It looks right" is not a metric. Build a labelled query set before optimising.
4. **Wrong K value**: Too small misses relevant docs; too large adds noise. Benchmark K=3, 5, 10, 20.
5. **No metadata filtering**: Production systems need category/date/permission filters -- build them early.
6. **Embedding model mismatch**: Use the same model for indexing and querying. Mismatched models produce random results.
7. **Not handling no-answer**: When the reader has no confident answer, say "I don't know" instead of guessing.

## Mini Challenge / Exercises

1. **Add metadata filtering**: Modify the query pipeline to filter by `category` (e.g. only search Kubernetes docs). Measure how filtering changes recall.
2. **Implement a GenerativeReader**: Replace the extractive reader with one that concatenates retrieved docs into a prompt and generates a natural language answer (use Ollama/Qwen).
3. **Evaluate at multiple K**: Run the evaluation at K=1, 3, 5, 10, 20. Plot Recall@K curves for each retriever.
4. **Add a ColBERT-style retriever**: Implement late interaction -- embed each token separately and compute MaxSim scores between query and document tokens.
5. **Build a feedback loop**: After each query, let the user mark which results were relevant. Use this to update relevance labels and retrain the query classifier.

## Final Summary & Key Takeaways

### What We Built
- **Full Haystack-style component system**: Component base class, typed I/O, pipeline DAG
- **InMemoryDocumentStore**: with BM25 index and embedding retrieval
- **3 preprocessing components**: DocumentCleaner, DocumentSplitter, DocumentEmbedder
- **3 retriever strategies**: BM25 (sparse), Embedding (dense), Hybrid (RRF fusion)
- **ExtractiveReader**: keyword-overlap answer extraction from retrieved passages
- **Pipeline class**: DAG execution with component wiring and execution tracing
- **QueryClassifier**: regex-based query routing to optimal retriever
- **AdaptivePipeline**: end-to-end query routing + retrieval + answer extraction
- **Evaluation framework**: Recall@K, Precision@K, MRR, nDCG@K, Token F1
- **30 labelled queries** across 4 types and 3 difficulty levels

### Haystack Production Equivalents

| This Notebook | Haystack Production |
|---|---|
| `Document` | `haystack.dataclasses.Document` |
| `Component` | `@component` decorator |
| `InMemoryDocumentStore` | `haystack.document_stores.in_memory.InMemoryDocumentStore` |
| `DocumentCleaner` | `haystack.components.preprocessors.DocumentCleaner` |
| `DocumentSplitter` | `haystack.components.preprocessors.DocumentSplitter` |
| `DocumentEmbedder` | `SentenceTransformersDocumentEmbedder` |
| `TextEmbedder` | `SentenceTransformersTextEmbedder` |
| `BM25Retriever` | `InMemoryBM25Retriever` |
| `EmbeddingRetriever` | `InMemoryEmbeddingRetriever` |
| `ExtractiveReader` | `ExtractiveReader(model="deepset/roberta-base-squad2")` |
| `Pipeline` | `haystack.Pipeline` |
| `QueryClassifier` | Custom `@component` with `ConditionalRouter` |

---
*Educational notebook -- part of the NLP / Information Retrieval portfolio.*